# 02 - Feature Engineering (lap time project)

Building the columns to predict lap time with. Same rule as before - only
use stuff we'd know before that lap happens.


In [1]:
import pandas as pd
import numpy as np


In [2]:
lap_times = pd.read_csv("lap_times.csv", na_values="\\N")
races = pd.read_csv("races.csv", na_values="\\N")
results = pd.read_csv("results.csv", na_values="\\N")
pit_stops = pd.read_csv("pit_stops.csv", na_values="\\N")


## merge everything into one lap-level table

In [ ]:
df = lap_times.merge(races[['raceId', 'year', 'round', 'circuitId']], on='raceId')
df = df.merge(results[['raceId', 'driverId', 'constructorId']], on=['raceId', 'driverId']) ## merging with 1st line.
df = df.sort_values(['year', 'round', 'driverId', 'lap']).reset_index(drop=True)
df['target'] = df['milliseconds']
df.shape
df

,raceId,driverId,lap,position,time,milliseconds,year,round,circuitId,constructorId,target
0,224,14,1,19,1:56.926,116926,1996,1,1,1,116926
1,224,14,2,18,1:41.839,101839,1996,1,1,1,101839
2,224,14,3,18,1:40.412,100412,1996,1,1,1,100412
3,224,14,4,18,1:39.572,99572,1996,1,1,1,99572
4,224,14,5,17,1:40.361,100361,1996,1,1,1,100361
...,...,...,...,...,...,...,...,...,...,...,...
589076,1144,862,53,15,1:29.717,89717,2024,24,24,214,89717
589077,1144,862,54,15,1:29.539,89539,2024,24,24,214,89539
589078,1144,862,55,15,1:29.647,89647,2024,24,24,214,89647
589079,1144,862,56,15,1:29.121,89121,2024,24,24,214,89121


## drop obvious outlier laps

laps that are more than 2x slower than the median lap time for that race
are almost certainly safety car laps, crashes, or laps with something
weird going on. keeping these in would confuse the model, so dropping
them. this is a rough filter, not perfect, but it gets rid of the most
extreme cases.

In [ ]:
race_median = df.groupby('raceId')['milliseconds'].transform('median')
print(race_median)
before_rows = len(df)

df = df[df['milliseconds'] <= race_median * 2] ## Keeps milliseconds which are less than race_median *2

after_rows = len(df)
print(f"dropped {before_rows - after_rows} rows out of {before_rows}")


0         97083.0
1         97083.0
2         97083.0
3         97083.0
4         97083.0
           ...   
589076    89756.0
589077    89756.0
589078    89756.0
589079    89756.0
589080    89756.0
Name: milliseconds, Length: 587985, dtype: float64
dropped 0 rows out of 587985


## laps since last pit stop (rough tyre wear proxy)

for each driver in each race, figure out how many laps it's been since
their last pit stop. if they haven't pitted yet, it's just the lap number
(still on their starting tyres). doing this with a loop, one race+driver
group at a time, so I can actually track "last pit lap" as I go through
each lap in order.

In [22]:
df['laps_since_pit'] = np.nan

for (race_id, driver_id), group in df.groupby(['raceId', 'driverId']):
    idx_list = group.sort_values('lap').index.tolist()

    # get this driver's pit stop laps in this race
    stops = pit_stops[(pit_stops['raceId'] == race_id) & (pit_stops['driverId'] == driver_id)]
    pit_laps = sorted(stops['lap'].tolist())

    last_pit_lap = 0  # 0 means "hasn't pitted yet, still on starting tyres"

    for idx in idx_list:
        current_lap = df.loc[idx, 'lap']

        # update last_pit_lap if we've passed a pit stop
        for p in pit_laps:
            if p <= current_lap:
                last_pit_lap = p

        df.loc[idx, 'laps_since_pit'] = current_lap - last_pit_lap


quick check on one race+driver to make sure this looks right

In [24]:
check = df[(df['raceId'] == df['raceId'].iloc[0])]
check_driver = check['driverId'].iloc[0]
check[check['driverId'] == check_driver][['lap', 'laps_since_pit']].head(20)


,lap,laps_since_pit
0,1,1.0
1,2,2.0
2,3,3.0
3,4,4.0
4,5,5.0
5,6,6.0
6,7,7.0
7,8,8.0
8,9,9.0
9,10,10.0


## build a race-level average pace first (needed for driver_form and constructor_form)

can't compute "recent pace" at the lap level directly, so first I build one
average lap time per driver per race, then use THAT to build rolling form,
same idea as the position prediction project.

In [7]:
race_pace = df.groupby(['raceId', 'driverId', 'constructorId', 'year', 'round'])['milliseconds'].mean().reset_index()
race_pace = race_pace.rename(columns={'milliseconds': 'race_avg_laptime'})
race_pace = race_pace.sort_values(['year', 'round']).reset_index(drop=True)

race_pace.head()


,raceId,driverId,constructorId,year,round,race_avg_laptime
0,224,14,1,1996,1,99797.708333
1,224,21,18,1996,1,101550.656250
2,224,22,17,1996,1,98117.655172
3,224,30,6,1996,1,99427.093750
4,224,35,3,1996,1,96698.465517


## driver's recent pace (avg lap time over their last 5 races, not counting this one)

In [8]:
race_pace['driver_form'] = np.nan

for driver_id in race_pace['driverId'].unique():
    driver_rows = race_pace[race_pace['driverId'] == driver_id]
    idx_list = driver_rows.index.tolist()
    past_paces = []

    for idx in idx_list:
        if len(past_paces) == 0:
            form_value = np.nan
        else:
            last_5 = past_paces[-5:]
            form_value = sum(last_5) / len(last_5)

        race_pace.loc[idx, 'driver_form'] = form_value
        past_paces.append(race_pace.loc[idx, 'race_avg_laptime'])


## constructor's recent pace (same idea, for the team)

In [9]:
race_pace['constructor_form'] = np.nan

for constructor_id in race_pace['constructorId'].unique():
    team_rows = race_pace[race_pace['constructorId'] == constructor_id]
    idx_list = team_rows.index.tolist()
    past_paces = []

    for idx in idx_list:
        if len(past_paces) == 0:
            form_value = np.nan
        else:
            last_5 = past_paces[-5:]
            form_value = sum(last_5) / len(last_5)

        race_pace.loc[idx, 'constructor_form'] = form_value
        past_paces.append(race_pace.loc[idx, 'race_avg_laptime'])


## circuit baseline pace

some tracks are just naturally faster or slower than others (short tracks
vs long tracks). using the average race pace at that circuit from PAST
races only (not including the current one).

In [10]:
race_pace = race_pace.merge(races[['raceId', 'circuitId']], on='raceId')
race_pace = race_pace.sort_values(['year', 'round']).reset_index(drop=True)

race_pace['circuit_baseline'] = np.nan

for circuit_id in race_pace['circuitId'].unique():
    circuit_rows = race_pace[race_pace['circuitId'] == circuit_id]
    idx_list = circuit_rows.index.tolist()
    past_paces = []

    for idx in idx_list:
        if len(past_paces) == 0:
            baseline_value = np.nan
        else:
            baseline_value = sum(past_paces) / len(past_paces)

        race_pace.loc[idx, 'circuit_baseline'] = baseline_value
        past_paces.append(race_pace.loc[idx, 'race_avg_laptime'])


## merge these race-level features back onto the lap-level table

In [11]:
df = df.merge(
    race_pace[['raceId', 'driverId', 'driver_form', 'constructor_form', 'circuit_baseline']],
    on=['raceId', 'driverId'],
    how='left'
)

df.shape


(587989, 15)

## final feature list

In [12]:
feature_cols = [
    'lap',
    'laps_since_pit',
    'driver_form',
    'constructor_form',
    'circuit_baseline',
]

target_col = 'target'

df[feature_cols + [target_col]].head()


,lap,laps_since_pit,driver_form,constructor_form,circuit_baseline,target
0,1,1.0,NaN,NaN,NaN,116926
1,2,2.0,NaN,NaN,NaN,101839
2,3,3.0,NaN,NaN,NaN,100412
3,4,4.0,NaN,NaN,NaN,99572
4,5,5.0,NaN,NaN,NaN,100361


## check missing values

In [13]:
df[feature_cols + [target_col]].isnull().sum()

lap                    0
laps_since_pit         0
driver_form         6558
constructor_form    1903
circuit_baseline    2111
target                 0
dtype: int64

In [14]:
# missing driver_form / constructor_form / circuit_baseline just means
# no history yet (driver's first 5 races, or first time at that circuit).
# will drop these rows when training the model later, same as before.

pct_missing = df['driver_form'].isnull().mean() * 100
print(f"{pct_missing:.1f}% of rows have no driver_form value yet")


1.1% of rows have no driver_form value yet


## save it

In [15]:
df.to_csv("processed_laptime_features.csv", index=False)
print("saved")


saved
